In [1]:
# Import necessary libraries for data analysis and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display all columns for better debugging and understanding
pd.set_option('display.max_columns', None)

In [2]:
# Load all datasets required for the analysis
# These represent different parts of the business (orders, customers, products, etc.)

orders = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
geo = pd.read_csv("olist_geolocation_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")

In [3]:
# Merge all datasets into a single analytical dataset
# This allows us to analyze relationships across customers, orders, products, and reviews

df = orders.merge(reviews, on='order_id') \
           .merge(order_items, on='order_id') \
           .merge(products, on='product_id') \
           .merge(customers, on='customer_id') \
           .merge(payments,on='order_id') \
           .merge(sellers,on='seller_id')                                         #This unified dataset ensures consistency and avoids repeated merging during analysis

In [4]:
geo_clean = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lng=('geolocation_lng', 'mean')
).reset_index()

In [5]:
df = df.merge(
    geo_clean,
    left_on='seller_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)

In [6]:
df = df.rename(columns={
    'lat': 'seller_lat',
    'lng': 'seller_lng'
})

In [7]:
df = df.merge(
    geo_clean,
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)

In [8]:
df = df.rename(columns={
    'lat': 'customer_lat',
    'lng': 'customer_lng'
})

In [9]:
df = df.drop(columns=[
    'geolocation_zip_code_prefix_x',
    'geolocation_zip_code_prefix_y'
])

In [10]:
# Convert timestamp columns to datetime format for time-based calculations
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

# Calculate total delivery time (from purchase to actual delivery)
df['delivery_time'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

# Calculate delivery gap (difference between actual and expected delivery)
df['delivery_gap'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

# Create delivery_delay feature (only positive delays, ignore early deliveries)
df['delivery_delay'] = df['delivery_gap'].apply(lambda x: x if x > 0 else 0)    #These features capture logistics performance, which is a key factor in customer satisfaction

In [11]:
# Calculate total revenue generated by each unique customer
# This helps identify high-value vs low-value customers

customer_revenue = df.groupby('customer_unique_id')['price'].sum()

# Summary statistics to understand distribution of customer spending
customer_revenue.describe()    #Reveals that most customers are one-time buyers, indicating low retention

count    94720.000000
mean       149.292666
std        248.710140
min          0.850000
25%         48.900000
50%         89.900000
75%        159.900000
max      13440.000000
Name: price, dtype: float64

In [12]:
# Analyze category-level performance in terms of revenue, order volume, and pricing

category_summary = df.groupby('product_category_name').agg(
    total_revenue=('price', 'sum'),     # total money generated by category
    order_count=('order_id', 'count'),  # number of items sold (volume)
    avg_price=('price', 'mean')         # average price per item
).sort_values('total_revenue', ascending=False)

category_summary.head()   #Helps identify which categories drive revenue and whether they are volume-driven or price-driven

,total_revenue,order_count,avg_price
product_category_name,,,
beleza_saude,1290883.52,9944,129.815318
relogios_presentes,1245783.11,6161,202.204692
cama_mesa_banho,1095770.05,11847,92.493462
esporte_lazer,1022489.29,8942,114.346823
informatica_acessorios,944992.54,8105,116.593774


In [13]:
# Understand delivery time distribution
df['delivery_time'].describe()

# Calculate percentage of delayed deliveries (only for delivered orders)
late_pct = (
    (df['delivery_delay'] > 0).sum() /
    df[df['order_status'] == 'delivered']['order_id'].count()
) * 100

late_pct     #Shows that most deliveries are on time, and delays are relatively infrequent

np.float64(6.41220975282738)

In [14]:
# Analyze how delivery time impacts customer satisfaction (review score)

df.groupby('review_score')['delivery_time'].mean()    #Higher delivery time is associated with lower review scores, indicating impact on satisfaction

review_score
1    19.092011
2    15.381893
3    13.552435
4    11.778330
5    10.203253
Name: delivery_time, dtype: float64

In [15]:
# Create binary features to capture high price and long delivery scenarios

df['high_price'] = df['price'] > df['price'].median()
df['long_delivery'] = df['delivery_time'] > df['delivery_time'].median()

# Combine both conditions to identify worst-case scenario
df['bad_combo'] = df['high_price'] & df['long_delivery']

# Compare average review scores for this combination
df.groupby('bad_combo')['review_score'].mean()   #High-priced products with long delivery times tend to receive lower customer ratings

bad_combo
False    4.093915
True     3.834049
Name: review_score, dtype: float64

In [16]:
# Check how review distribution changes under bad conditions

df.groupby('bad_combo')['review_score'].value_counts(normalize=True)   #Shows a drop in 5-star reviews and increase in 1-star reviews under bad conditions

bad_combo  review_score
False      5               0.587071
           4               0.184643
           1               0.116129
           3               0.079543
           2               0.032614
True       5               0.494272
           4               0.206703
           1               0.159709
           3               0.097535
           2               0.041780
Name: proportion, dtype: float64

In [17]:
# Measure how sensitive each category is to delivery delays
# Correlation helps identify strength of relationship

category_corr = df.groupby('product_category_name').apply(
    lambda x: x['delivery_time'].corr(x['review_score'])
).sort_values()

category_corr.head()    #Different categories react differently to delivery delays, indicating varying customer expectations

C:\ProgramData\anaconda3\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\ProgramData\anaconda3\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\Yashika Sharma\AppData\Local\Temp\ipykernel_2412\1852372698.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  category_corr = df.groupby('product_category_name').apply(


product_category_name
fashion_underwear_e_moda_praia   -0.648352
moveis_colchao_e_estofado        -0.545906
musica                           -0.538691
casa_conforto_2                  -0.502938
tablets_impressao_imagem         -0.483729
dtype: float64

In [18]:
# Compute average product price per product
product_price = df.groupby('product_id')['price'].mean().reset_index()

# Merge back to main dataset
df = df.merge(product_price, on='product_id', suffixes=('', '_product_avg'))

# Final category-level summary combining multiple dimensions
category_final = df.groupby('product_category_name').agg(
    avg_review=('review_score', 'mean'),
    avg_delivery=('delivery_time', 'mean'),
    avg_product_price=('price_product_avg', 'mean')
).sort_values('avg_review')

category_final.head()   #Provides a holistic view of category performance across satisfaction, delivery, and pricing

,avg_review,avg_delivery,avg_product_price
product_category_name,,,
seguros_e_servicos,2.500000,15.000000,141.645000
pc_gamer,3.100000,8.555556,167.594000
fraldas_higiene,3.256410,10.243243,40.194615
portateis_cozinha_e_preparadores_de_alimentos,3.266667,7.785714,264.568667
moveis_escritorio,3.526791,20.480549,160.388094


## Key Insights:
Customer retention is very low (~3%), indicating most customers are one-time buyers
Delivery performance affects customer satisfaction but is not the only factor
High price combined with long delivery time significantly reduces review scores
Customer sensitivity to delivery varies across product categories
Customer satisfaction is influenced by multiple interacting factors (delivery, price, category)

In [19]:
# Average review by payment type
df.groupby('payment_type')['review_score'].mean().sort_values()

payment_type
voucher        4.003184
boleto         4.024699
credit_card    4.032871
debit_card     4.156028
Name: review_score, dtype: float64

In [20]:
# Payment type distribution across review scores
df.groupby('payment_type')['review_score'].value_counts(normalize=True)

payment_type  review_score
boleto        5               0.559361
              4               0.193387
              1               0.126867
              3               0.086709
              2               0.033676
credit_card   5               0.565973
              4               0.189413
              1               0.126953
              3               0.083079
              2               0.034582
debit_card    5               0.608156
              4               0.180851
              1               0.104610
              3               0.074468
              2               0.031915
voucher       5               0.556192
              4               0.187202
              1               0.126711
              3               0.086915
              2               0.042980
Name: proportion, dtype: float64

In [21]:
df.groupby('payment_installments')['review_score'].mean()  #does installments effect score

payment_installments
0     5.000000
1     4.069787
2     4.063342
3     3.999147
4     4.007420
5     4.018324
6     4.011711
7     4.006590
8     3.929728
9     4.072626
10    3.847245
11    3.454545
12    3.829268
13    4.000000
14    4.375000
15    3.692308
16    3.857143
17    3.571429
18    3.526316
20    3.850000
21    4.000000
22    1.000000
23    3.000000
24    2.705882
Name: review_score, dtype: float64

In [22]:
# Number of items per order
order_size = df.groupby('order_id')['order_item_id'].count().reset_index()
order_size.columns = ['order_id', 'num_items']

df = df.merge(order_size, on='order_id')

In [23]:
df.groupby('num_items')['review_score'].mean()

num_items
1     4.161723
2     3.782095
3     3.594165
4     3.550061
5     3.487903
6     3.355839
7     3.574468
8     4.000000
9     3.692308
10    3.187500
11    4.272727
12    3.642857
13    3.500000
14    2.600000
15    5.000000
16    4.750000
19    1.000000
20    1.500000
21    2.000000
22    3.000000
24    2.800000
26    5.000000
29    1.000000
38    5.000000
63    5.000000
Name: review_score, dtype: float64

In [24]:
# Bucket orders
df['order_size_bucket'] = pd.cut(
    df['num_items'],
    bins=[0,1,3,5,10,100],
    labels=['1','2-3','4-5','6-10','10+']
)

df.groupby('order_size_bucket')['review_score'].mean()

C:\Users\Yashika Sharma\AppData\Local\Temp\ipykernel_2412\1855672320.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('order_size_bucket')['review_score'].mean()


order_size_bucket
1       4.161723
2-3     3.746206
4-5     3.532994
6-10    3.468872
10+     3.529745
Name: review_score, dtype: float64

In [25]:
seller_perf = df.groupby('seller_id').agg(
    avg_delivery=('delivery_time', 'mean'),
    avg_review=('review_score', 'mean'),
    order_count=('order_id', 'count')
).sort_values('avg_delivery', ascending=False)

seller_perf.sample(10)

,avg_delivery,avg_review,order_count
seller_id,,,
3f9be91358837bff69df67edfa3e42e8,9.428571,4.428571,7
1e483cc5c76fef08d3ca05f9a8af7d01,7.636364,4.166667,12
9bf11dfc0bec77e5a23028043c3c5a8f,31.000000,1.000000,1
dd7ddc04e1b6c2c614352b383efe2d36,14.655844,3.748387,155
87d3c3aeb3ead335511b3ce315eb341e,5.714286,3.500000,14
5343d0649eca2a983820bfe93fc4d17e,10.871795,3.542373,118
4559697a8f7e637227c2eeaed843baff,12.309524,4.088889,45
2a50b7ee5aebecc6fd0ff9784a4747d6,39.000000,1.000000,2
079d295dcbf06ee8bb1b65ba964eb2b6,6.300000,4.800000,10


In [26]:
seller_perf.head(10)   # more the delivery time less the review score not always but maximum

,avg_delivery,avg_review,order_count
seller_id,,,
df683dfda87bf71ac3fc63063fba369d,189.000000,1.000000,1
8e670472e453ba34a379331513d6aab1,86.000000,1.000000,1
586a871d4f1221763fddb6ceefdeb95e,68.000000,1.000000,2
4fb41dff7c50136976d1a5cf004a42e2,66.333333,4.000000,3
8629a7efec1aab257e58cda559f03ba7,59.000000,1.000000,1
3da38366e7bd9baf6369071f782ecdf0,53.000000,1.000000,1
eebb3372362aa9a46975164bed19a7e7,52.250000,2.285714,7
8c3b533c63cca56240f94f1e3a6b18ef,49.000000,1.333333,6
244b04680fdbded0acf5aebd9c92b44a,49.000000,1.000000,2


In [27]:
df['same_state'] = df['seller_state'] == df['customer_state']

In [28]:
df.groupby('same_state')['delivery_time'].mean()   # location effecting delivery time

same_state
False    14.529446
True      7.472388
Name: delivery_time, dtype: float64

In [29]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    
    return R * c

df['distance_km'] = df.apply(
    lambda x: haversine(x['seller_lat'], x['seller_lng'],
                        x['customer_lat'], x['customer_lng']),
    axis=1
)

In [30]:
df[['distance_km', 'delivery_time']].corr()

,distance_km,delivery_time
distance_km,1.000000,0.392605
delivery_time,0.392605,1.000000


In [31]:
df.groupby(pd.qcut(df['distance_km'], 5))['delivery_time'].mean()

C:\Users\Yashika Sharma\AppData\Local\Temp\ipykernel_2412\3220574699.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.qcut(df['distance_km'], 5))['delivery_time'].mean()


distance_km
(-0.001, 116.0]         6.201017
(116.0, 347.502]       10.418898
(347.502, 527.718]     11.843269
(527.718, 872.419]     13.757984
(872.419, 8677.912]    17.627481
Name: delivery_time, dtype: float64

In [32]:
df['distance_bucket'] = pd.qcut(df['distance_km'], 5)

In [33]:
seller_perf = df.groupby(['seller_id', 'distance_bucket']).agg(
    avg_delivery=('delivery_time', 'mean'),
    order_count=('order_id', 'count')
).reset_index()

C:\Users\Yashika Sharma\AppData\Local\Temp\ipykernel_2412\265068226.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  seller_perf = df.groupby(['seller_id', 'distance_bucket']).agg(


In [34]:
seller_perf.sort_values('avg_delivery', ascending=False).head(10)  # to check if seller inefficient or is it distance

,seller_id,distance_bucket,avg_delivery,order_count
13473,df683dfda87bf71ac3fc63063fba369d,"(527.718, 872.419]",189.0,1
13969,e83c76265fc54bf41eac728805e4da77,"(872.419, 8677.912]",188.0,1
4779,4fb41dff7c50136976d1a5cf004a42e2,"(872.419, 8677.912]",168.0,1
14399,eebb3372362aa9a46975164bed19a7e7,"(872.419, 8677.912]",166.0,2
7520,7a91bf945c6fae0779f1c61ce97fe45c,"(-0.001, 116.0]",115.0,1
13530,e09887ca8c7bf8a4621ce481820414ef,"(-0.001, 116.0]",105.0,1
12315,cb41bfbcbda0aea354a834ab222f9a59,"(-0.001, 116.0]",99.0,3
6772,6f1a1263039c76e68f40a8e536b1da6a,"(347.502, 527.718]",94.0,1
8709,8e670472e453ba34a379331513d6aab1,"(872.419, 8677.912]",86.0,1
3950,427165bf50f8ca07efc7bdc2bfcf1688,"(-0.001, 116.0]",86.0,1


In [35]:
df.groupby('review_score')['order_id'].count()

review_score
1    14854
2     4085
3     9840
4    22286
5    66264
Name: order_id, dtype: int64

In [36]:
num_duplicates = df.groupby('review_score')['customer_unique_id'].apply(lambda x: x.duplicated().sum())
print(num_duplicates)

review_score
1     4171
2     1036
3     1841
4     3524
5    10912
Name: customer_unique_id, dtype: int64


In [37]:
df_cust = df[['customer_unique_id', 'order_id', 'review_score']].drop_duplicates()

In [38]:
order_count = df_cust.groupby('customer_unique_id')['order_id'].nunique().reset_index()
order_count.columns = ['customer_unique_id', 'num_orders']

In [39]:
order_count['is_repeat'] = order_count['num_orders'] > 1

In [40]:
cust_review = df_cust.groupby('customer_unique_id')['review_score'].first().reset_index()

In [41]:
cust_final = cust_review.merge(order_count, on='customer_unique_id')

In [42]:
cust_final.groupby('review_score')['is_repeat'].mean()

review_score
1    0.028601
2    0.030425
3    0.030647
4    0.026872
5    0.031701
Name: is_repeat, dtype: float64

## insights
Customer retention in this dataset is largely independent of review score, indicating that satisfaction alone does not drive repeat purchases. This suggests that retention is influenced by external or business factors beyond customer experience.

### FINAL PROBLEM STATEMENT 

I aim to predict customer satisfaction (review score) based on delivery performance, product characteristics, and order behavior, in order to identify and reduce negative customer experiences.

In [43]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'review_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'seller_zip_code_prefix',
       'seller_city', 'seller_state', 'seller_lat', 'seller_lng',
       'customer_lat', 'customer_lng', 'delivery_time', 'd

In [52]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [45]:
df['target'] = df['review_score'].apply(lambda x: 0 if x <= 3 else 1)

In [46]:
df['target'].value_counts(normalize=True)

target
1    0.754715
0    0.245285
Name: proportion, dtype: float64

In [47]:
num_cols = [
    'delivery_time',
    'delivery_delay',
    'price',
    'freight_value',
    'payment_installments',
    'num_items',
    'distance_km',
    'product_weight_g',
    'product_photos_qty',
    'price_product_avg'
]

In [48]:
cat_cols = [
    'product_category_name',
    'payment_type',
    'order_size_bucket',
    'seller_state',
    'same_state'
]

In [49]:
df.isnull().sum()

order_id                              0
customer_id                           0
order_status                          0
order_purchase_timestamp              0
order_approved_at                    15
order_delivered_carrier_date       1235
order_delivered_customer_date      2471
order_estimated_delivery_date         0
review_id                             0
review_score                          0
review_comment_title             103437
review_comment_message            67650
review_creation_date                  0
review_answer_timestamp               0
order_item_id                         0
product_id                            0
seller_id                             0
shipping_limit_date                   0
price                                 0
freight_value                         0
product_category_name              1695
product_name_lenght                1695
product_description_lenght         1695
product_photos_qty                 1695
product_weight_g                     20


In [50]:
X = df[num_cols + cat_cols]
y = df['target']

In [51]:
from sklearn.model_selection import train_test_split 
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

In [54]:
num_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scalar',StandardScaler())
])

In [56]:
cat_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

In [57]:
preprocessor=ColumnTransformer([
    ('num',num_pipeline,num_cols),
    ('cat',cat_pipeline,cat_cols)
])

In [58]:
from sklearn.linear_model import LogisticRegression
model=Pipeline([
    ('preprocessor',preprocessor),
    ('classifier',LogisticRegression(max_iter=1000))
])

In [59]:
model.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [60]:
from sklearn.metrics import classification_report
y_pred=model.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.76      0.16      0.26      5756
           1       0.78      0.98      0.87     17710

    accuracy                           0.78     23466
   macro avg       0.77      0.57      0.57     23466
weighted avg       0.78      0.78      0.72     23466

